%md
# Notebook 03 – OMOP Mapping

### Why OMOP Mapping?

Healthcare data from different source systems often follows different formats and schemas, making it difficult to perform unified analytics.

The OMOP Common Data Model (CDM) provides a standardized schema that enables consistent storage, querying, reporting, and machine learning across healthcare datasets.

### Objective

The objective of this notebook is to transform the cleaned Silver layer datasets into OMOP-compliant tables by mapping healthcare entities to the appropriate OMOP CDM structure.

### Input Tables

This notebook reads the following Silver layer tables:

- silver.patient_silver_dlt
- silver.encounter_silver_dlt
- silver.condition_silver_dlt
- silver.observation_silver_dlt
- silver.medication_request_silver_dlt

In [0]:
%run ./00_UnityCatalog_Bootstrap_&_Audit_log

In [0]:
from datetime import datetime

start_time = datetime.now()

%md
### 1. Load Silver Layer Tables

The cleaned Silver datasets are loaded and used as the source for OMOP mapping.

Each Silver table represents a standardized healthcare entity that will be transformed into its corresponding OMOP table.

In [0]:
patient_df = spark.read.table(
    "db_healthcare_kl.silver.patient_silver_dlt"
)

encounter_df = spark.read.table(
    "db_healthcare_kl.silver.encounter_silver_dlt"
)

condition_df = spark.read.table(
    "db_healthcare_kl.silver.condition_silver_dlt"
)

observation_df = spark.read.table(
    "db_healthcare_kl.silver.observation_silver_dlt"
)

medication_df = spark.read.table(
    "db_healthcare_kl.silver.medication_request_silver_dlt"
)

In [0]:
patient_df.printSchema()

%md
### 2. Map Patient Data to OMOP Person

Patient records are transformed into the OMOP **Person** table.

Examples of field mappings include:

- patient_id → person_id
- gender → gender_concept_id
- birth_date → date_of_birth
- full_name → person_name

Gender values are converted into standard OMOP concept IDs to maintain consistency across datasets.

In [0]:
from pyspark.sql.functions import col, when, year, to_date

In [0]:
person_df = patient_df.select(
    col("patient_id").alias("person_id"),

    when(col("gender") == "male", 8507)
    .when(col("gender") == "female", 8532)
    .otherwise(0)
    .alias("gender_concept_id"),

    col("birth_date").alias("date_of_birth"),

    col("full_name").alias("person_name")
)

In [0]:
person_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_healthcare_kl.omop.person_data")

In [0]:
display(person_df)

%md
### 3. Map Encounter Data to Visit Occurrence

Encounter information is mapped to the OMOP **Visit Occurrence** table.

Examples include:

- encounter_id → visit_occurrence_id
- patient_reference → person_id
- start_date → visit_start_date
- end_date → visit_end_date
- patient_class → visit_type
- reason → visit_reason

In [0]:
encounter_df.printSchema()

In [0]:
visit_occurrence_df = encounter_df.select(
    col("encounter_id").alias("visit_occurrence_id"),
    col("patient_reference").alias("person_id"),
    col("start_date").alias("visit_start_date"),
    col("end_date").alias("visit_end_date"),
    col("patient_class").alias("visit_type"),
    col("reason").alias("visit_reason")
)

In [0]:
visit_occurrence_df.write\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_healthcare_kl.omop.visit_occurrence")

%md
### 4. Map Clinical Conditions

Clinical diagnosis records are transformed into the OMOP **Condition Occurrence** table.

This mapping standardizes diagnosis information while preserving patient relationships and encounter references.

In [0]:
condition_df.printSchema()

In [0]:
condition_occurrence_df = condition_df.select(
    col("Condition_Id").alias("condition_occurrence_id"),
    col("Patient_Reference").alias("person_id"),
    col("Encounter_Reference").alias("visit_occurrence_id"),
    col("Condition_Code").alias("condition_source_value"),
    col("Condition_Name").alias("condition_name"),
    to_date(col("Recorded_Date")).alias("condition_start_date"),
    to_date(col("Onset_Date")).alias("condition_end_date"),
    col("Clinical_Status").alias("clinical_status"),
    col("Verification_Status").alias("verification_status")
)

In [0]:
condition_occurrence_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_healthcare_kl.omop.condition_occurrence")

In [0]:
observation_df.printSchema()

%md
### 5. Map Clinical Observations

Observation records such as laboratory results and clinical measurements are mapped into the OMOP **Observation** table.

Each observation remains linked to its corresponding patient and encounter.

In [0]:
from pyspark.sql.functions import expr

observation_omop_df = observation_df.select(
    col("Observation_Id").alias("observation_id"),
    col("Patient_Reference").alias("person_id"),
    col("Encounter_Reference").alias("visit_occurrence_id"),
    col("Observation_Name").alias("observation_source_value"),
    col("Observation_Date").alias("observation_date"),
    expr("try_cast(Value as double)").alias("value_as_number"),
    col("Value").alias("value_source_value"),
    col("Unit").alias("unit_source_value"),
    col("Status").alias("observation_status"),
    col("Issued_Date").alias("issued_date")
)

In [0]:
observation_omop_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_healthcare_kl.omop.observation_occurrence")

%md
### 6. Map Medication Records

Medication requests are transformed into the OMOP **Drug Exposure** table.

This standardizes medication information for analytics and downstream machine learning applications.

In [0]:
medication_df.printSchema()

In [0]:
drug_exposure_df = medication_df.select(
    col("MedicationRequest_Id").alias("drug_exposure_id"),
    col("Patient_Reference").alias("person_id"),
    col("Encounter_Reference").alias("visit_occurrence_id"),
    col("Medication_Code").alias("drug_source_value"),
    col("Medication_Name").alias("drug_name"),
    to_date(col("Authored_On")).alias("drug_exposure_start_date"),
    col("Status").alias("drug_status"),
    col("Intent").alias("drug_intent"),
    col("Requester_Reference").alias("provider_id"),
    col("Requester_Name").alias("provider_name")
)

In [0]:
drug_exposure_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("db_healthcare_kl.omop.drug_exposure")

%md
### 7. Save OMOP Tables

After mapping, each OMOP table is stored as a Delta table.

The generated tables include:

- omop.person_data
- omop.visit_occurrence
- omop.condition_occurrence
- omop.observation
- omop.drug_exposure

These standardized tables serve as the foundation for the Gold layer and machine learning feature engineering.

In [0]:
end_time = datetime.now()

condition_occurence = spark.read.table("db_healthcare_kl.omop.condition_occurrence")
observation_occurrence = spark.read.table("db_healthcare_kl.omop.observation_occurrence")
drug_exposure = spark.read.table("db_healthcare_kl.omop.drug_exposure")
person_data = spark.read.table("db_healthcare_kl.omop.person_data")
visit_occurrence = spark.read.table("db_healthcare_kl.omop.visit_occurrence") 

record_count = condition_occurence.count() + observation_occurrence.count() + drug_exposure.count() + person_data.count() + visit_occurrence.count()



log_pipeline_run(
    pipeline_name="Healthcare Lakehouse",
    layer="Bronze",
    status="SUCCESS",
    records_processed=record_count,
    start_time=start_time,
    end_time=end_time
)